# CNAE Baseline Analysis

This notebook fetches vulnerability reports created by the Trivy Operator from Kubernetes and generates data used for the baseline evaluation.

This notebook creates the following files:
- `data/vulnerabilityreports/*.yaml` - Raw vulnerability reports
- `data/otel_vulnerability_summary.csv` - Vulnerability counts per image
- `data/otel_vulnerabilities.csv` - List of found vulnerabilities with source type
- `data/otel_source_categorization.csv` - Vulnerability counts by source type
- `data/otel_unique_vulnerabilities.csv`- unique CVEs
- `data/otel_unique_vulnerability_severity_distribution.csv` - Number of unique CVE count per severity

In [1]:
import numpy as np
import pandas as pd
from kubernetes import client, config
import csv
import io
import os
import yaml

## Fetch Vulnerability Reports

In [2]:
## This cell was generated by Claude Sonnet 4
def fetch_vulnerability_reports(namespace="my-otel-demo"):
    """
    Fetch vulnerability reports from Kubernetes
    """
    
    # Load Kubernetes config (from ~/.kube/config or in-cluster config)
    try:
        config.load_kube_config()  # For local development
    except:
        config.load_incluster_config()  # For running inside a pod
    
    # Create a client for custom resources
    api_client = client.ApiClient()
    custom_api = client.CustomObjectsApi(api_client)
    
    try:
        # Get vulnerability reports from the specified namespace
        vulnerability_reports = custom_api.list_namespaced_custom_object(
            group="aquasecurity.github.io",
            version="v1alpha1",
            namespace=namespace,
            plural="vulnerabilityreports"
        )

        # Save each report as a yaml file
        os.makedirs("data/vulnerabilityreports", exist_ok=True)
        for report in vulnerability_reports.get('items', []):
            report_name = report['metadata']['name']
            with open(f"data/vulnerabilityreports/{report_name}.yaml", 'w') as file:
                yaml.dump(report, file)
        
        return vulnerability_reports.get('items', [])
        
        
    except Exception as e:
        print(f"Error fetching vulnerability reports: {e}")
        return None

In [3]:
vulnerability_reports = fetch_vulnerability_reports()
otel_vulnerability_reports = [item for item in vulnerability_reports if item.get("report",{}).get("artifact",{}).get("repository") == "open-telemetry/demo"]

# Analysis

## Vulnerability counts per image by severity

In [4]:
## This cell was generated by Claude Sonnet 4
otel_summary_df = pd.DataFrame([
    {
        'Service': item.get('metadata', {}).get('name', ''),
        'Image': f"{item.get('report', {}).get('artifact', {}).get('repository', '')}:{item.get('report', {}).get('artifact', {}).get('tag', '')}",
        'Critical': item.get('report', {}).get('summary', {}).get('criticalCount', 0),
        'High': item.get('report', {}).get('summary', {}).get('highCount', 0),
        'Medium': item.get('report', {}).get('summary', {}).get('mediumCount', 0),
        'Low': item.get('report', {}).get('summary', {}).get('lowCount', 0),
        'Unknown': item.get('report', {}).get('summary', {}).get('unknownCount', 0),
        'creationTimestamp': item.get('metadata', {}).get('creationTimestamp', '')
    }
    for item in otel_vulnerability_reports
])
otel_summary_df.to_csv("data/otel_vulnerability_summary.csv", index=False)
otel_summary_df

,Service,Image,Critical,High,Medium,Low,Unknown,creationTimestamp
0,replicaset-frontend-b68ff86cb-frontend,open-telemetry/demo:2.0.2-frontend,1,0,1,2,0,2025-06-11T10:09:57Z


## Vulnerability counts per source type by severity

Categorize vulnerabilities by source type (OS/Application):

In [5]:
all_vulnerabilities = []
for item in otel_vulnerability_reports:
    report = item.get("report", {})
    vulnerabilities = report.get("vulnerabilities", [])
    all_vulnerabilities.extend(vulnerabilities)

otel_vulnerabilities_df = pd.DataFrame(all_vulnerabilities)


In [6]:
# This function was generated by ChatGPT o4-mini-high
def detect_library_type(purl: str) -> str:
    """
    Determines based on the Package PURL whether it is an OS package or an
    application library.
    We recognize OS packages by known packaging formats or by the presence
    of a 'distro' qualifier.
    """
    # Known OS prefixes (Debian, RPM, Alpine, Windows MSI)
    os_prefixes = ("pkg:deb/", "pkg:rpm/", "pkg:apk/", "pkg:msi/")
    # If the PURL starts with an OS prefix or contains a 'distro=' parameter,
    # then we count it as an OS package.
    if any(purl.startswith(pref) for pref in os_prefixes) or "distro=" in purl:
        return "os"
    return "app"

In [7]:
otel_vulnerabilities_df['source_type'] = otel_vulnerabilities_df['packagePURL'].apply(detect_library_type)
otel_vulnerabilities_df.to_csv("data/otel_vulnerabilities.csv", index=False)
otel_vulnerabilities_df

,fixedVersion,installedVersion,lastModifiedDate,links,packagePURL,primaryLink,publishedDate,resource,score,severity,target,title,vulnerabilityID,source_type
0,"7.26.10, 8.0.0-alpha.17",7.22.5,2025-03-11T20:15:18Z,[],pkg:npm/%40babel/runtime@7.22.5,https://avd.aquasec.com/nvd/cve-2025-27789,2025-03-11T20:15:18Z,@babel/runtime,6.2,MEDIUM,,Babel is a compiler for writing next generatio...,CVE-2025-27789,app
1,,2.0.1,2025-06-09T19:15:25Z,[],pkg:npm/brace-expansion@2.0.1,https://avd.aquasec.com/nvd/cve-2025-5889,2025-06-09T19:15:25Z,brace-expansion,3.1,LOW,,brace-expansion Regular Expression Denial of S...,CVE-2025-5889,app
2,"13.5.9, 14.2.25, 15.2.3, 12.3.5",15.2.1,2025-04-08T14:15:33Z,[],pkg:npm/next@15.2.1,https://avd.aquasec.com/nvd/cve-2025-29927,2025-03-21T15:15:42Z,next,9.1,CRITICAL,,nextjs: Authorization Bypass in Next.js Middle...,CVE-2025-29927,app
3,15.2.2,15.2.1,2025-05-30T16:31:03Z,[],pkg:npm/next@15.2.1,https://avd.aquasec.com/nvd/cve-2025-48068,2025-05-30T04:15:48Z,next,4.3,LOW,,next.js: Information exposure in Next.js dev s...,CVE-2025-48068,app


In [8]:
categorization_df = otel_vulnerabilities_df.groupby(['source_type','severity']).size().reset_index(name='count')
categorization_df.to_csv("data/otel_source_categorization.csv", index=False)
categorization_df

,source_type,severity,count
0,app,CRITICAL,1
1,app,LOW,2
2,app,MEDIUM,1


## Number of unique CVEs

Some CVEs appear in several severity categories. We have chosen to place them in the most severe category.

In [10]:
## This cell was generated by Claude Sonnet 4
# Extract unique vulnerabilities using highest severity approach

# Define severity hierarchy (higher number = more severe)
severity_hierarchy = {
    'UNKNOWN': 0,
    'LOW': 1, 
    'MEDIUM': 2,
    'HIGH': 3,
    'CRITICAL': 4
}

# Add severity rank to the dataframe
otel_vulnerabilities_df['severity_rank'] = otel_vulnerabilities_df['severity'].map(severity_hierarchy)

# For each vulnerabilityID, keep only the row with the highest severity
unique_vulnerabilities_df = otel_vulnerabilities_df.loc[
    otel_vulnerabilities_df.groupby('vulnerabilityID')['severity_rank'].idxmax()
].drop('severity_rank', axis=1)

# Save unique vulnerabilities to CSV
unique_vulnerabilities_df.to_csv("data/otel_unique_vulnerabilities.csv", index=False)

# Show severity distribution of unique vulnerabilities as DataFrame
severity_counts = unique_vulnerabilities_df['severity'].value_counts()
unique_severity_distribution = pd.DataFrame({
    'Critical': [severity_counts.get('CRITICAL', 0)],
    'High': [severity_counts.get('HIGH', 0)],
    'Medium': [severity_counts.get('MEDIUM', 0)],
    'Low': [severity_counts.get('LOW', 0)],
    'Unknown': [severity_counts.get('UNKNOWN', 0)]
})

unique_severity_distribution.to_csv("data/otel_unique_vulnerability_severity_distribution.csv", index=False)
unique_severity_distribution

,Critical,High,Medium,Low,Unknown
0,1,0,1,2,0
